In [1]:
import os

PROJECT = "/content/warp_reduction"
SRC = os.path.join(PROJECT, "src")

os.makedirs(SRC, exist_ok=True)

print("Working in:", PROJECT)

Working in: /content/warp_reduction


In [5]:
%%writefile /content/warp_reduction/src/first_kernel.cu
#include <iostream>
#include <cuda_runtime.h>

// GPU function: runs on the GPU
__global__ void addTen(int* data, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n) {
        data[i] = data[i] *2;
    }
}

int main() {
    const int n = 5;
    int h_data[n] = {1, 2, 3, 4, 5};

    int* d_data;

    // Allocate memory on the GPU
    cudaMalloc(&d_data, n * sizeof(int));

    // Copy input from CPU to GPU
    cudaMemcpy(
        d_data, h_data, n * sizeof(int),
        cudaMemcpyHostToDevice
    );

    // Launch the GPU kernel
    int threadsPerBlock = 32;
    int blocks = (n + threadsPerBlock - 1)
                 / threadsPerBlock;

    addTen<<<blocks, threadsPerBlock>>>(d_data, n);

    // Wait for GPU work to finish
    cudaDeviceSynchronize();

    // Copy result from GPU to CPU
    cudaMemcpy(
        h_data, d_data, n * sizeof(int),
        cudaMemcpyDeviceToHost
    );

    // Print result
    std::cout << "Output: ";
    for (int i = 0; i < n; i++) {
        std::cout << h_data[i] << " ";
    }
    std::cout << std::endl;

    // Free GPU memory
    cudaFree(d_data);

    return 0;
}

Overwriting /content/warp_reduction/src/first_kernel.cu


In [6]:
!nvcc /content/warp_reduction/src/first_kernel.cu \
    -o /content/warp_reduction/src/first_kernel

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [7]:
!/content/warp_reduction/src/first_kernel

Output: 2 4 6 8 10 
